# 📦 Solución Completa: Base de Datos FIA Chirimoyo
## Arquitectura ETL + SQL Server

## 📂 Contenido del Paquete

### Scripts SQL Server (Ejecutar en orden)
```
1. 01_crear_esquemas.sql
   └─ Crea esquemas: stg, dim, fact
   
2. 02_crear_tablas.sql
   └─ Crea todas las tablas con constraints y índices
   └─ Especialmente importante para estudiantes:
      • dim.tratamiento (catálogo)
      • stg.mediciones_raw (staging/auditoría)
      • fact.mediciones_climaticas (fact principal)
      • fact.dpv (fact especializada)
      • fact.horas_frio (fact especializada)
   
3. 03_procedimientos_etl.sql
   └─ Crea procedures almacenados:
      • sp_cargar_tratamientos()
      • sp_cargar_mediciones_raw(@ruta_csv)
      • sp_transformar_a_fact()
   
4. 04_workflow_carga_completo.sql
   └─ Workflow completo + consultas de validación
   └─ Ideal para ejecutar después de cargar datos
```

### Scripts Python
```
etl_chirimoyo_v3.py
└─ Pipeline de transformación de datos
└─ Lee: .xlsx original
└─ Genera: CSVs con formato chileno
└─ Genera reporte de calidad
└─ Uso: python3 etl_chirimoyo_v3.py
```

### Datos (CSVs listos para cargar)
```
dim_tratamiento.csv (2 registros)
└─ Catálogo de tratamientos

mediciones_completo.csv (1,860 registros)
└─ Todos los datos campo + invernadero

mediciones_campo.csv (930 registros)
└─ Solo tratamiento CAMPO

mediciones_invernadero.csv (930 registros)
└─ Solo tratamiento INVERNADERO

Formato:
✓ Delimitador: ; (punto y coma)
✓ Codificación: UTF-8 con BOM
✓ Decimal: , (coma)
✓ Líneas: CRLF
```

### Documentación
```phyton
GUIA_PEDAGOGICA_CHIRIMOYO.md
└─ Guía completa para estudiantes
└─ Explica arquitectura, conceptos, ejemplos
└─ Troubleshooting y ejercicios

README.md (este archivo)
└─ In

## 🚀 Inicio Rápido (5 minutos)

### 1. Crear Base de Datos en SQL Server

En **SQL Server Management Studio (SSMS)**:

```sql
-- Crear base de datos (si no existe)
CREATE DATABASE FIA_Chirimoyo
GO
USE FIA_Chirimoyo
GO

-- Ejecutar scripts en orden:
-- 1. 01_crear_esquemas.sql
-- 2. 02_crear_tablas.sql
-- 3. 03_procedimientos_etl.sql
```

### 2. Cargar Datos

En **SSMS**, ejecutar:

```sql
USE FIA_Chirimoyo
GO

-- Cargar catálogos
EXEC sp_cargar_tratamientos @debug = 1

-- Cargar mediciones (ajustar ruta)
EXEC sp_cargar_mediciones_raw 
    @ruta_csv = 'C:\ruta\a\mediciones_completo.csv',
    @debug = 1

-- Transformar a fact tables
EXEC sp_transformar_a_fact @debug = 1
```

### 3. Validar

```sql
-- Contar registros
SELECT COUNT(*) AS Total FROM fact.mediciones_climaticas
-- Debe mostrar: 1860

-- Comparar Campo vs Invernadero
SELECT t.nombre_tratamiento, COUNT(*) AS Registros
FROM fact.mediciones_climaticas mc
INNER JOIN dim.tratamiento t ON mc.id_tratamiento = t.id_tratamiento
GROUP BY t.nombre_tratamiento
-- Debe mostrar:
-- Campo          930
-- Invernadero    930
```

## 📖 Estructura para Docencia

### Clase 1: Conceptos ETL
- **Tema:** ¿Cómo van datos de sensores a base de datos?
- **Materiales:** `GUIA_PEDAGOGICA_CHIRIMOYO.md` (secciones Arquitectura + Flujo)
- **Demo:** Mostrar `etl_chirimoyo_v3.py` ejecutándose

### Clase 2: Diseño de Base de Datos Agrícola
- **Tema:** Esquemas, tablas, constraints
- **Materiales:** `01_crear_esquemas.sql`, `02_crear_tablas.sql`
- **Ejercicio:** Estudiantes crean sus propias tablas para otro cultivo

### Clase 3: SQL Avanzado
- **Tema:** Procedures, validación, transformación
- **Materiales:** `03_procedimientos_etl.sql`
- **Ejercicio:** Modificar sp_transformar_a_fact para agregar más validaciones

### Clase 4: Análisis de Datos Agrícolas
- **Tema:** Comparar campos experimentales
- **Materiales:** `04_workflow_carga_completo.sql`
- **Análisis:** Horas frío (campo: 255.1 vs invernadero: 225.8)

## 🔍 Análisis Clave Generado

### Hallazgo 1: Horas Frío
```
Campo:       255.1 horas acumuladas
Invernadero: 225.8 horas acumuladas
Diferencia:  11.5% menos en invernadero

INTERPRETACIÓN:
Chirimoya requiere ~400-600 horas < 7.2°C para florecer uniformemente.
El invernadero protege del frío, lo que puede afectar uniformidad de floración.
```

### Hallazgo 2: Temperaturas
```
Campo promedio:       14.62°C (±3.21)
Invernadero promedio: 15.88°C (±3.06)

INTERPRETACIÓN:
Invernadero es ~1.3°C más cálido. Más estable (menor desviación).
```

### Hallazgo 3: Humedad
```
Campo:       83.4% (±5.6)
Invernadero: 81.2% (±6.4)

INTERPRETACIÓN:
Ambos altos. Invernadero ligeramente más seco pero más variable.
```

### Anomalía Detectada
```
Fecha: 2026-04-18 (Campo)
temp_min=11.7, temp_max=95.4 ← ¡IMPOSIBLE!

Causa probable: Error de sensor o transcripción
Acción: Marcar como inválido en fact.mediciones_climaticas
```

## 🛠️ Personalización para Tus Datos

Si quieres adaptar esto a **otro experimento o cultivo**:

### Opción 1: Cambiar nombres de tratamientos
En `01_crear_esquemas.sql`, cambiar:
```sql
INSERT INTO dim.tratamiento VALUES ('Mi_Tratamiento_1', 'Descripción...')
INSERT INTO dim.tratamiento VALUES ('Mi_Tratamiento_2', 'Descripción...')
```

### Opción 2: Agregar variables nuevas
En `02_crear_tablas.sql`, agregar columnas a `fact.mediciones_climaticas`:
```sql
-- Ejemplo: agregar radiación PAR
par_umol_m2_s DECIMAL(8,2),
```

### Opción 3: Cambiar rango de validación
En `02_crear_tablas.sql`, modificar CHECKs:
```sql
-- Para otro cultivo con temperaturas diferentes:
CONSTRAINT ck_fact_temp_max_rango 
    CHECK (temp_max >= 0 AND temp_max <= 55)  -- cambiar según cultivo
```

### Opción 4: Crear fact tables adicionales
Para otra variable crítica (ej: humedad del suelo):
```sql
CREATE TABLE fact.humedad_suelo (
    id_hs INT IDENTITY(1,1) PRIMARY KEY,
    id_tratamiento INT NOT NULL REFERENCES dim.tratamiento,
    fecha DATE NOT NULL,
    humedad_suelo_pct DECIMAL(5,2) NOT NULL,
    profundidad_cm INT,
    -- ...
)
```

## 📊 Exporte de Datos para Análisis Posterior

### Exportar a Python/Pandas
```python
import pandas as pd
import pyodbc

conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=localhost\SQLEXPRESS;'
    'Database=FIA_Chirimoyo;'
    'Trusted_Connection=yes'
)

df = pd.read_sql("""
    SELECT mc.fecha, mc.temp_media, mc.hr_media, 
           t.nombre_tratamiento
    FROM fact.mediciones_climaticas mc
    INNER JOIN dim.tratamiento t ON mc.id_tratamiento = t.id_tratamiento
""", conn)

df.to_csv('para_analisis.csv', index=False)
```

### Exportar a Excel
```sql
-- Copiar resultados de esta consulta a Excel:
SELECT 
    mc.fecha,
    t.nombre_tratamiento,
    mc.temp_min, mc.temp_media, mc.temp_max,
    mc.hr_min, mc.hr_media, mc.hr_max,
    fd.dpv_kpa,
    fh.hf_acumuladas
FROM fact.mediciones_climaticas mc
LEFT JOIN fact.dpv fd ON mc.id_tratamiento = fd.id_tratamiento 
    AND mc.fecha = fd.fecha
LEFT JOIN fact.horas_frio fh ON mc.id_tratamiento = fh.id_tratamiento 
    AND mc.fecha = fh.fecha
ORDER BY mc.fecha
```

## ✅ Checklist de Implementación

- [ ] Descargar todos los archivos
- [ ] Crear base de datos `FIA_Chirimoyo` en SQL Server
- [ ] Ejecutar `01_crear_esquemas.sql`
- [ ] Ejecutar `02_crear_tablas.sql`
- [ ] Ejecutar `03_procedimientos_etl.sql`
- [ ] Copiar CSVs a carpeta accesible desde SQL Server
- [ ] Ejecutar procedures de carga
- [ ] Validar con consultas de verificación
- [ ] Ejecutar `04_workflow_carga_completo.sql` para análisis
- [ ] Leer `GUIA_PEDAGOGICA_CHIRIMOYO.md` para entender diseño

---

## 🚨 Problemas Comunes

| Problema | Solución |
|----------|----------|
| "Archivo no encontrado" en BULK INSERT | Verificar ruta completa del CSV, usar rutas UNC en red |
| "Error de codificación" | Asegurar que CSV esté en UTF-8 con BOM |
| "FK constraint violation" | Verificar que tratamientos en CSV coincidan con dim.tratamiento |
| "Numeric value out of range" | Datos violan CHECK constraint (ej: temp > 50°C) |
| Caracteres raros (acentos) | CSV debe estar en UTF-8 con BOM, no ANSI |

---


---

## 📚 Complementos Sugeridos

1. **Visualización:** Tableau/Power BI sobre fact tables
2. **Análisis estadístico:** R o Python para comparar tratamientos
3. **Monitoreo:** Crear dashboard en Excel/Sheets con datos actualizados
4. **Predicción:** Modelo ML para predecir próximas horas frío

---

## 📝 Notas Finales

- **Versión:** 3.0 (junio 2026)
- **Cultivo:** Chirimoya (Annona cherimola)
- **Tratamientos:** Campo vs Invernadero
- **Período:** Nov 2023 - Abr 2026
- **Total registros:** 1,860